In [71]:
## Load my files ##
import sys
sys.path.append('..')
from utils import get_sequence

## Load standard files ##
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim.lr_scheduler as lr_scheduler
from torch import from_numpy as tnsr
from scipy.stats import bernoulli
import torch.nn as nn
import numpy as np
import pandas as pd
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist as dist
from sklearn.metrics.pairwise import cosine_similarity
from scipy.signal import find_peaks
from scipy.spatial.distance import cosine
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import torch.nn.functional as F

In [72]:
n_community = 2
n_members = 3

tokens = []

for ii in range(n_community*n_members+1):
    tokens.append(
        chr(ord('A')+ii)
    )

In [73]:
class brain(nn.Module):
    def __init__(self, input_size, hidden_wake_size, hidden_sleep_size, context_output_size, num_layers=1, num_layers_sleep=1):
        super(brain, self).__init__()

        self.rnn = nn.RNN(input_size+context_output_size, hidden_wake_size, num_layers, nonlinearity='relu', batch_first=True)
        self.wake_fc = nn.Linear(hidden_wake_size, len(tokens))
        # self.context_fc = nn.Linear(hidden_wake_size, sleep_input_size)

        self.sleep_rnn = nn.RNN(hidden_wake_size, hidden_sleep_size, num_layers_sleep, nonlinearity='relu', batch_first=True)
        self.context_out = nn.Linear(hidden_sleep_size, context_output_size)
        self.sleep_fc = nn.Linear(hidden_sleep_size, hidden_wake_size)

        self.context_output_size = context_output_size

    def wake(self):
        self.rnn.requires_grad = True 
        self.wake_fc.requires_grad = True

        self.sleep_rnn.requires_grad = False
        self.context_out.requires_grad = True
        self.sleep_fc.requires_grad = False

    def sleep(self):
        self.rnn.requires_grad = False 
        self.wake_fc.requires_grad = False

        self.sleep_rnn.requires_grad = True
        self.context_out.requires_grad = False
        self.sleep_fc.requires_grad = True
        
    def zero_sleep_weights(self):
        for name, param in self.sleep_rnn.named_parameters():
            nn.init.zeros_(param) 

    def forward(self, x, hw=None, hs=None):
        if hw != None:
            out, hs = self.sleep_rnn(hw, hs)
            sleep_out = self.context_out(out)
        else:
            sleep_out = torch.zeros((1,x.size(1),self.context_output_size))
        # print(sleep_out.shape) 
        x = torch.cat((x,sleep_out), dim=2)
        
        
        out, hw = self.rnn(x, hw)
        out = self.wake_fc(out[:,-1,:])


        return out, sleep_out, hw, hs
 


In [ ]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size=7, num_layers=1):
        super(RNN, self).__init__()

        self.rnn = nn.RNN(input_size, hidden_size, num_layers, nonlinearity='relu', batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x, h=None):
        out, h = self.rnn(x, h)
        out = self.fc(out[:,-1,:])

        return out, h

In [74]:
class Dataset_converter(Dataset):
    def __init__(self, data, working_memory=1, short_term_memory=8):
        
        one_hot_encoded = np.zeros((len(data), len(tokens)), dtype=float)
        for ii, token in enumerate(data):
            one_hot_encoded[ii,ord(token)-65] = 1
        
        self.X = np.zeros((((len(data)-working_memory-short_term_memory)), short_term_memory, len(tokens)*working_memory))
        self.y = np.zeros((((len(data)-working_memory-short_term_memory)), len(tokens)))

        for ii in range(self.X.shape[0]):
            for jj in range(self.X.shape[1]):
                for kk in range(working_memory):
                    self.X[ii,jj,kk*len(tokens):(kk+1)*len(tokens)] = \
                    one_hot_encoded[ii+jj+kk,:]
                    
            self.y[ii] = \
                one_hot_encoded[ii+jj+kk+1,:]

        self.X = tnsr(self.X).float()
        self.y = tnsr(self.y).float()

    def __getitem__(self, index):
        return self.X[index], self.y[index]

    def __len__(self):
        return self.X.shape[0]

In [80]:
### initial training ###
total_samples = 50000
working_memory = 1
short_term_memory = 1
hidden_wake_size = 40
hidden_sleep_size = 40
context_output_size = 5
num_layers_wake = 1
num_layers_sleep = 1
output_sleep = len(tokens)
input_size = len(tokens)*working_memory
lr = 1e-3
test_acc = []
c = []

data = get_sequence(total_samples, n_community, n_members, train_percent=1.0)#gen_seq(total_samples) #

data_set = Dataset_converter(data, working_memory, short_term_memory)
train_loader = DataLoader(data_set, batch_size=1, shuffle=False)

network1 = brain(input_size, hidden_wake_size, hidden_sleep_size, context_output_size, num_layers=num_layers_wake, num_layers_sleep=num_layers_sleep)
network1.zero_sleep_weights()
network1.wake()

optimizer = torch.optim.SGD(network1.parameters(), lr=lr, momentum=0.95)
criterion = torch.nn.CrossEntropyLoss()

total = 0
correct = np.zeros(1000,dtype=float)
for X, y in train_loader:
    optimizer.zero_grad()

    if total == 0:
        predicted_y, _, hw, hs = network1(X)
    else:
        predicted_y, _, hw, hs = network1(X, hw=mem, hs=mem_)
    
    # print(predicted_y.shape, y.shape)
    loss = criterion(predicted_y, y)
    #loss_repel = 1e-1 * network1.repulsion_loss(margin=1.0)
    
    # loss = loss #+ loss_repel
    
    loss.backward(retain_graph=True)
    optimizer.step()

    with torch.no_grad():
        mem=hw.clone()

        if hs != None:
            mem_=hs.clone()
        else:
            mem_ = None

        c.append(hw[0][0])
        true_y = y.argmax(axis=1)
        estimated_y = predicted_y.argmax(axis=1)

        total += 1
        if true_y == estimated_y:
                correct[total%1000] = 1
        else:
            correct[total%1000] = 0

        test_acc.append(
            np.sum(correct)/total if total<1000 else np.sum(correct)/1000
        )
        if total%1000 == 0:
            print(f'Iter : {total+1}, loss: {loss:.4f}, accuracy: {test_acc[-1]:.4f}')


Iter : 1001, loss: 1.9696, accuracy: 0.2470
Iter : 2001, loss: 1.2811, accuracy: 0.4400
Iter : 3001, loss: 2.2499, accuracy: 0.5770
Iter : 4001, loss: 2.5104, accuracy: 0.6520
Iter : 5001, loss: 1.1103, accuracy: 0.6690
Iter : 6001, loss: 1.0274, accuracy: 0.6650
Iter : 7001, loss: 1.6506, accuracy: 0.6660
Iter : 8001, loss: 1.9379, accuracy: 0.6490
Iter : 9001, loss: 1.4638, accuracy: 0.6580
Iter : 10001, loss: 1.7781, accuracy: 0.6550
Iter : 11001, loss: 1.6102, accuracy: 0.6720
Iter : 12001, loss: 1.5545, accuracy: 0.6590
Iter : 13001, loss: 1.7060, accuracy: 0.6450
Iter : 14001, loss: 1.4746, accuracy: 0.6800
Iter : 15001, loss: 2.0548, accuracy: 0.6740
Iter : 16001, loss: 1.8764, accuracy: 0.6690
Iter : 17001, loss: 1.7930, accuracy: 0.6610
Iter : 18001, loss: 1.8836, accuracy: 0.6730
Iter : 19001, loss: 1.6983, accuracy: 0.6750
Iter : 20001, loss: 1.7530, accuracy: 0.6540
Iter : 21001, loss: 1.4223, accuracy: 0.6760
Iter : 22001, loss: 1.6840, accuracy: 0.6700
Iter : 23001, loss:

In [76]:
data[-20:]

'EFDGBACGEFDGBCAGFDEG'

In [77]:
hs

tensor([[[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]]], grad_fn=<StackBackward0>)

In [22]:
for ii in range(20):
    print(c[-ii], data[-ii])

tensor([0.0000, 0.0146, 0.0000, 0.0174, 0.1500, 0.0000, 0.0670, 0.0000, 0.0000,
        0.0728, 0.0000, 0.0285, 0.0542, 0.0136, 0.0902, 0.0000, 0.2578, 0.2938,
        0.1591, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0992, 0.0530, 0.1794,
        0.0361, 0.0970, 0.0071, 0.0000, 0.0000, 0.1795, 0.0000, 0.0211, 0.0000,
        0.3480, 0.0000, 0.1001, 0.0000], requires_grad=True) A
tensor([0.0000, 0.6173, 0.0000, 0.0000, 0.4602, 0.0656, 0.3901, 0.0000, 0.0000,
        0.8807, 0.0000, 0.9906, 1.1830, 0.2612, 1.0249, 1.8257, 1.4836, 0.4839,
        0.0000, 1.8575, 0.7775, 0.5942, 0.4613, 0.0000, 0.0000, 0.0000, 0.8007,
        0.5935, 1.3590, 0.0000, 0.0000, 0.0180, 0.7994, 0.8959, 0.0000, 0.4001,
        0.1160, 2.1203, 1.1036, 0.0000], requires_grad=True) G
tensor([0.0000, 0.8340, 0.0000, 0.0000, 0.3072, 0.2370, 0.1008, 0.0000, 0.0000,
        0.7832, 0.0000, 1.4969, 0.6466, 0.2352, 0.5766, 0.9573, 1.1232, 0.7433,
        0.2715, 2.6779, 0.7857, 0.8034, 0.6446, 0.0000, 0.0000, 0.0000, 0.

In [82]:
total_samples = 50
l = len(c)
D = np.zeros((total_samples, total_samples), dtype=float)

for ii in range(total_samples):
    for jj in range(total_samples):
        D[ii,jj] = cosine(c[l-ii-1].detach().numpy(), c[l-jj-1].detach().numpy()) 
        print(data[l-ii-1], data[l-jj-1], D[ii,jj])

D D 1.285816153551167e-08
D F 0.31068434228687614
D G 0.8873984286032398
D A 0.7556196739052875
D C 0.7669179707881378
D B 0.8190477512695471
D G 0.8692229655313374
D D 0.34273580271319815
D F 0.4146253028630885
D E 0.6269512222255137
D G 0.8819663939628253
D C 0.5468005445346369
D A 0.7021001426687485
D B 0.8187760811759213
D G 0.869222968034358
D D 0.345100538048581
D F 0.42053934527182746
D E 0.6322632868201014
D G 0.8692229686687405
D D 0.39281881379260974
D E 0.4182556245160397
D F 0.3148150739098735
D G 0.8853610754705293
D A 0.7608930214480316
D C 0.7694098091976129
D B 0.8119131289079022
D G 0.8926758443396021
D C 0.5597775902249515
D B 0.6679364028103563
D A 0.7434301519758536
D G 0.8693386054023555
D F 0.3434262939401068
D D 0.3516908952824266
D E 0.6266194443506481
D G 0.8858031875095556
D A 0.761190126000233
D C 0.7679509984446045
D B 0.8111140052934029
D G 0.8938729869094572
D C 0.5575566666013483
D B 0.6687313946577202
D A 0.7412749981731692
D G 0.8702150347665609
D F 0.4

In [48]:
D

array([[2.32856496e-08, 2.07265260e-03, 1.57299895e-01, 1.92165015e-01,
        3.14233280e-02, 3.17906631e-02, 1.08587563e-01, 2.53117544e-01],
       [2.07265260e-03, 0.00000000e+00, 1.88830848e-01, 2.12786373e-01,
        4.22840054e-02, 3.99942949e-02, 1.34117044e-01, 2.56191782e-01],
       [1.57299895e-01, 1.88830848e-01, 2.11798856e-09, 1.69117273e-01,
        1.31069536e-01, 1.33580997e-01, 2.60504235e-02, 2.52264275e-01],
       [1.92165015e-01, 2.12786373e-01, 1.69117273e-01, 0.00000000e+00,
        1.16378974e-01, 2.78109323e-01, 1.05473124e-01, 1.06674866e-01],
       [3.14233280e-02, 4.22840054e-02, 1.31069536e-01, 1.16378974e-01,
        1.58367839e-08, 4.78101340e-02, 5.62914780e-02, 2.01744972e-01],
       [3.17906631e-02, 3.99942949e-02, 1.33580997e-01, 2.78109323e-01,
        4.78101340e-02, 4.95708463e-09, 9.55245821e-02, 3.36495603e-01],
       [1.08587563e-01, 1.34117044e-01, 2.60504235e-02, 1.05473124e-01,
        5.62914780e-02, 9.55245821e-02, 0.00000000e+00, 1.

In [37]:
data[l-8:l]

'DGBCAGFD'

In [38]:
c[l-ii-1][0][0]

tensor([-0.7345, -0.5629,  1.0458, -0.4893,  0.4876])